# Convolutional Networks — a hands-on MNIST playground

*Companion to Chapter 9 of Goodfellow, Bengio & Courville — "Convolutional Networks."*

A CNN bakes the structure of grid data into its wiring: connections are **local**, one small **filter** is **shared** across every position, and **pooling** tolerates small shifts. Those constraints slash parameters, respect spatial layout, and gave vision its modern era. This notebook turns each lab of the playground into a runnable MNIST experiment.

| # | Station | What we run on MNIST |
|---|---------|----------------------|
| 1 | Why CNNs? | Dense vs conv parameter counts |
| 2 | The convolution operation | Cross-correlation from scratch; edge/blur feature maps |
| 3 | Sparse · shared · equivariant | Equivariance: shift in → feature map shifts |
| 4 | Pooling | Max/avg; translation-invariance test |
| 5 | Neuroscience & Gabor | Gabor bank + orientation tuning curve |
| 6 | Infinitely strong prior | Dense vs locally-connected vs conv weight matrices |
| 7 | Variants | Stride/padding/dilation output-size formula, verified |
| 8 | Structured outputs & channels | Multi-channel maps; per-pixel segmentation |
| 9 | Efficient algorithms | Separable kernel = outer product; FFT convolution |
| 10 | Random features & BatchNorm | Random filters find edges; BN steadies training |
| 11 | Computer vision & ImageNet | Train a CNN vs an MLP; feature hierarchy |

> **The one idea:** convolution and pooling are **infinitely strong priors** — local connectivity, shared weights, and small-shift invariance, assumed before any data — and they happen to be true for images.

**Runtime:** CPU works; a GPU (Runtime → Change runtime type → GPU) speeds up the training stations (5, 8, 10, 11). Run top to bottom.


## Setup — data and shared helpers

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

transform = transforms.ToTensor()                       # keep images as [1,28,28] in [0,1]
train_full = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_full  = datasets.MNIST('./data', train=False, download=True, transform=transform)

def make_loader(n, bs=128, seed=0, train=True):
    ds = train_full if train else test_full
    idx = np.random.RandomState(seed).choice(len(ds), n, replace=False)
    return DataLoader(Subset(ds, idx), batch_size=bs, shuffle=train)

train_loader = make_loader(8000, 128)
val_loader   = make_loader(4000, 256, train=False)
sample_img = test_full[0][0][0].numpy()                 # a 28x28 array for the hand-coded demos
print('train 8000 | val 4000 | sample image', sample_img.shape)

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval(); c=t=0
    for xb,yb in loader:
        xb,yb=xb.to(device),yb.to(device)
        c += (model(xb).argmax(1)==yb).sum().item(); t += yb.size(0)
    return c/t

def train_clf(model, epochs=6, lr=1e-3, loader=None, record=False):
    loader = loader or train_loader
    model.to(device); opt = torch.optim.Adam(model.parameters(), lr); losses=[]
    for ep in range(epochs):
        model.train()
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(); loss=F.cross_entropy(model(xb),yb); loss.backward(); opt.step()
            if record: losses.append(loss.item())
    return (accuracy(model,val_loader), losses)

## Station 1 — Why convolutional networks?

A dense layer on a `256×256×3` photo ties every one of ~197k inputs to every unit, so a 1000-unit layer needs ~197M weights — and none of them know that neighbouring pixels are related or that a shifted cat is still a cat. Convolution fixes the **parameter explosion** by sliding one small filter everywhere. Even on tiny MNIST the gap is stark.


In [ ]:
H = W = 28
dense_layer   = H*W * 1000                    # 784 -> 1000 fully-connected weights
conv_layer    = 1*32 * 3*3                    # 32 filters, 3x3, 1 input channel
big_dense     = 256*256*3 * 1000              # the same layer on a 256x256 RGB photo
print(f'MNIST dense 784->1000     : {dense_layer:>12,} weights')
print(f'MNIST conv  1->32 (3x3)   : {conv_layer:>12,} weights')
print(f'ratio                     : {dense_layer/conv_layer:>12,.0f}x fewer for conv')
print(f'\n256x256x3 dense ->1000    : {big_dense:>12,} weights (~197M) — and it grows with image size')
print('A conv layer reuses the same filter everywhere, so its weight count does NOT grow with image size.')

## Station 2 — The convolution operation

Slide a small **kernel** over the input; at each position record the weighted sum of the pixels it covers. That number says how strongly the local patch matches the kernel's pattern; sweeping gives a **feature map**. Libraries don't flip the kernel, so this is technically **cross-correlation**. We implement it from scratch, apply classic kernels to a digit, and check it against PyTorch.


In [ ]:
def conv2d_np(img, ker):                       # valid cross-correlation
    kh, kw = ker.shape; oh, ow = img.shape[0]-kh+1, img.shape[1]-kw+1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i,j] = (img[i:i+kh, j:j+kw] * ker).sum()
    return out

kernels = {
    'identity': np.array([[0,0,0],[0,1,0],[0,0,0]], float),
    'edge (Sobel-x)': np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float),
    'edge (Laplacian)': np.array([[0,1,0],[1,-4,1],[0,1,0]], float),
    'blur (box)': np.ones((3,3))/9,
}
fig, ax = plt.subplots(1, len(kernels)+1, figsize=(13,3))
ax[0].imshow(sample_img, cmap='gray'); ax[0].set_title('input'); ax[0].axis('off')
for k,(name,ker) in enumerate(kernels.items()):
    ax[k+1].imshow(conv2d_np(sample_img, ker), cmap='gray'); ax[k+1].set_title(name, fontsize=9); ax[k+1].axis('off')
plt.suptitle('Station 2 — feature maps for different kernels'); plt.tight_layout(); plt.show()

# cross-check against PyTorch F.conv2d
x = torch.tensor(sample_img).view(1,1,28,28).float()
w = torch.tensor(kernels['edge (Sobel-x)']).view(1,1,3,3).float()
torch_out = F.conv2d(x, w)[0,0].numpy()
print('scratch vs torch max abs diff:', np.abs(conv2d_np(sample_img, kernels['edge (Sobel-x)']) - torch_out).max())
print(f'valid output size for 28x28 with 3x3 kernel: {torch_out.shape} (= 28-3+1 = 26)')

## Station 3 — Sparse, shared, equivariant

Three mechanisms do the work. **Sparse interactions**: each output reads only a small patch. **Parameter sharing**: the same kernel runs at every position. **Equivariance to translation**: shift the input and the feature map shifts identically — `conv(shift(x)) = shift(conv(x))`. We verify equivariance directly and count the parameter savings.


In [ ]:
ker = kernels['edge (Laplacian)']
shift = 3
shifted_then_conv = conv2d_np(np.roll(sample_img, shift, axis=1), ker)
conv_then_shifted = np.roll(conv2d_np(sample_img, ker), shift, axis=1)
# compare on the interior (ignore the wrapped-around border columns)
inner = (slice(None), slice(shift, -shift))
diff = np.abs(shifted_then_conv[inner] - conv_then_shifted[inner]).max()

fig, ax = plt.subplots(1,3, figsize=(10,3))
ax[0].imshow(conv2d_np(sample_img, ker), cmap='gray'); ax[0].set_title('conv(x)'); ax[0].axis('off')
ax[1].imshow(shifted_then_conv, cmap='gray'); ax[1].set_title('conv(shift(x))'); ax[1].axis('off')
ax[2].imshow(conv_then_shifted, cmap='gray'); ax[2].set_title('shift(conv(x))'); ax[2].axis('off')
plt.suptitle('Station 3 — equivariance: the two right images match'); plt.tight_layout(); plt.show()
print(f'max difference on the interior: {diff:.2e}  -> conv(shift(x)) == shift(conv(x))')

# parameter savings (1-D illustration): m inputs -> n outputs
m, n, k = 784, 784, 9
print(f'\ndense would need m*n = {m*n:,} weights; convolution needs k = {k} weights.')

## Station 4 — Pooling

A conv stage is convolve → nonlinearity → **pool**. Pooling replaces each neighbourhood with a summary (max or average). Because a summary barely changes when the patch shifts by a pixel, pooling gives approximate **translation invariance** — you learn *whether* a feature is present, not exactly *where* — and striding shrinks the map.


In [ ]:
def pool_np(x, k=2, mode='max'):
    H, W = x.shape; oh, ow = H//k, W//k; out = np.zeros((oh,ow))
    for i in range(oh):
        for j in range(ow):
            patch = x[i*k:(i+1)*k, j*k:(j+1)*k]
            out[i,j] = patch.max() if mode=='max' else patch.mean()
    return out

feat = conv2d_np(sample_img, kernels['edge (Sobel-x)'])
feat = feat[:24,:24]                                    # crop to a multiple of the pool window
shifted = np.roll(sample_img, 1, axis=1)
feat_shift = conv2d_np(shifted, kernels['edge (Sobel-x)'])[:24,:24]

for mode in ['max','avg']:
    d_raw  = np.abs(feat - feat_shift).mean()
    d_pool = np.abs(pool_np(feat,2,mode) - pool_np(feat_shift,2,mode)).mean()
    print(f'{mode}-pool: mean change after 1px shift  raw {d_raw:.4f}  ->  pooled {d_pool:.4f}')

fig, ax = plt.subplots(1,3, figsize=(9,3))
ax[0].imshow(feat, cmap='gray'); ax[0].set_title('feature map (24x24)'); ax[0].axis('off')
ax[1].imshow(pool_np(feat,2,'max'), cmap='gray'); ax[1].set_title('max-pool -> 12x12'); ax[1].axis('off')
ax[2].imshow(pool_np(feat,2,'avg'), cmap='gray'); ax[2].set_title('avg-pool -> 12x12'); ax[2].axis('off')
plt.suptitle('Station 4 — pooling downsamples and tolerates small shifts'); plt.tight_layout(); plt.show()

## Station 5 — The neuroscientific basis (Gabor filters)

Hubel & Wiesel found V1 **simple cells** that fire for edges at a particular orientation (≈ an oriented filter applied everywhere = convolution) and **complex cells** tolerant to small shifts (≈ pooling). The first-layer filters a CNN *learns* look like **Gabor functions** — oriented edge detectors. We build a Gabor bank and measure its orientation tuning.


In [ ]:
def gabor(theta, f=3, sigma=4.0, size=21):
    c = size//2; y, x = np.mgrid[-c:c+1, -c:c+1]
    xr = x*np.cos(theta) + y*np.sin(theta)
    return np.exp(-(x**2+y**2)/(2*sigma**2)) * np.cos(2*np.pi*f*xr/size)

def grating(theta, f=3, size=21):
    c = size//2; y, x = np.mgrid[-c:c+1, -c:c+1]
    return np.cos(2*np.pi*f*(x*np.cos(theta)+y*np.sin(theta))/size)

orients = np.linspace(0, np.pi, 12, endpoint=False)
stim_theta = np.deg2rad(40)                              # show a 40-degree grating
stim = grating(stim_theta)
responses = [np.sum(gabor(o)*stim) for o in orients]
best = orients[int(np.argmax(responses))]

fig, ax = plt.subplots(1,3, figsize=(12,3.2))
ax[0].imshow(stim, cmap='gray'); ax[0].set_title('stimulus grating (40 deg)'); ax[0].axis('off')
ax[1].imshow(gabor(best), cmap='gray'); ax[1].set_title(f'best-matching Gabor ({np.rad2deg(best):.0f} deg)'); ax[1].axis('off')
ax[2].plot(np.rad2deg(orients), responses, 'o-', color='crimson'); ax[2].axvline(40, ls='--', color='gray')
ax[2].set_title('orientation tuning curve'); ax[2].set_xlabel('filter orientation (deg)'); ax[2].set_ylabel('response')
plt.tight_layout(); plt.show()
print('The bank responds most to the filter whose orientation matches the stimulus — a V1 simple cell.')

## Station 6 — Convolution as an infinitely strong prior

A convolutional layer is just a fully-connected layer with an **infinitely strong prior**: weights are *zero outside the receptive field* and *shared across positions*. A **locally connected** layer keeps locality but drops sharing. We build the three effective weight matrices (1-D, 8 inputs → 6 outputs, kernel 3) and count free parameters.


In [ ]:
n_in, k = 8, 3; n_out = n_in - k + 1
dense = np.random.RandomState(0).randn(n_out, n_in)
local = np.zeros((n_out, n_in)); conv = np.zeros((n_out, n_in))
shared = np.array([1.0, -2.0, 1.0])                    # one shared kernel for the conv row
for i in range(n_out):
    local[i, i:i+k] = np.random.RandomState(i).randn(k)  # local but each row independent
    conv[i, i:i+k]  = shared                              # local AND tied

fig, ax = plt.subplots(1,3, figsize=(11,3.2))
for a,(name,M) in zip(ax, [('dense', dense), ('locally connected', local), ('convolution', conv)]):
    a.imshow(M, cmap='RdBu', vmin=-2, vmax=2); a.set_title(name); a.set_xlabel('input'); a.set_ylabel('output')
plt.suptitle('Station 6 — the prior zeros-out and ties weights'); plt.tight_layout(); plt.show()
print(f'free parameters  dense {n_out*n_in}  |  locally connected {n_out*k}  |  convolution {k}')

## Station 7 — Variants: stride, padding, dilation

Real layers add knobs. **Stride** `s` skips positions (downsamples); **padding** `p` adds a border so output needn't shrink; **dilation** `d` spreads the kernel's taps to enlarge the receptive field with no extra weights. Output size along one axis:

$$o=\left\lfloor\frac{n+2p-k_{\text{eff}}}{s}\right\rfloor+1,\qquad k_{\text{eff}}=d(k-1)+1.$$

We verify the formula against PyTorch for several settings.


In [ ]:
def out_size(n,k,s,p,d): return (n + 2*p - (d*(k-1)+1))//s + 1
x = torch.randn(1,1,28,28)
print(f"{'k s p d':>10} | {'formula':>7} | {'torch':>5} | match")
for (k,s,p,d) in [(3,1,0,1),(3,1,1,1),(3,2,1,1),(5,1,0,1),(3,1,2,2),(3,3,0,1)]:
    o = F.conv2d(x, torch.randn(1,1,k,k), stride=s, padding=p, dilation=d).shape[-1]
    f = out_size(28,k,s,p,d)
    print(f"{k} {s} {p} {d}".rjust(10) + f" | {f:>7} | {o:>5} | {'OK' if o==f else 'FAIL'}")
print('\nStride shrinks the output; padding grows it back; dilation widens the receptive field for free.')

## Station 8 — Structured outputs & channels

Convolution preserves spatial layout, so a CNN can emit a **structured output** as large as the input — e.g. **semantic segmentation**, a label per pixel. And a kernel always spans every **channel**: its weight tensor is `(out_ch × in_ch × kH × kW)`. We show multi-channel feature maps, then train a tiny fully-convolutional net to segment the digit's foreground.


In [ ]:
conv = nn.Conv2d(1, 8, 3, padding=1)                    # 1 input channel -> 8 output channels
with torch.no_grad(): fmaps = conv(torch.tensor(sample_img).view(1,1,28,28).float())[0]
print('kernel weight tensor shape (out,in,kH,kW):', tuple(conv.weight.shape))
fig, ax = plt.subplots(1,8, figsize=(12,1.8))
for i in range(8):
    ax[i].imshow(fmaps[i].detach(), cmap='gray'); ax[i].axis('off'); ax[i].set_title(f'ch{i}', fontsize=8)
plt.suptitle('Station 8 — one conv layer produces 8 feature maps (channels)'); plt.show()

In [ ]:
# Semantic segmentation: predict the foreground mask (same H x W as the input).
class FCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(),
                                 nn.Conv2d(16,16,3,padding=1), nn.ReLU(),
                                 nn.Conv2d(16,1,3,padding=1))
    def forward(self, x): return self.net(x)

torch.manual_seed(0); fcn = FCN().to(device); opt = torch.optim.Adam(fcn.parameters(), 1e-3)
for ep in range(4):
    for xb,_ in train_loader:
        xb = xb.to(device); target = (xb > 0.5).float()      # foreground mask
        opt.zero_grad(); F.binary_cross_entropy_with_logits(fcn(xb), target).backward(); opt.step()

xb,_ = next(iter(val_loader)); x = xb[:1].to(device)
with torch.no_grad(): pred = torch.sigmoid(fcn(x))[0,0].cpu()
fig, ax = plt.subplots(1,3, figsize=(8,3))
ax[0].imshow(x[0,0].cpu(), cmap='gray'); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(pred, cmap='gray'); ax[1].set_title('predicted mask'); ax[1].axis('off')
ax[2].imshow((x[0,0].cpu()>0.5), cmap='gray'); ax[2].set_title('target mask'); ax[2].axis('off')
plt.suptitle('Station 8 — per-pixel structured output (segmentation)'); plt.tight_layout(); plt.show()

## Station 9 — Efficient convolution algorithms

A **separable** kernel factors into an outer product of two 1-D kernels, so a `k×k` filter costs `2k` multiplies per pixel instead of `k²`. Large kernels can go through the **FFT**, where convolution becomes point-wise multiplication. These change *how fast*, not *what* is computed — we verify both.


In [ ]:
# Separable: a Gaussian blur = outer product of two 1-D Gaussians.
g1 = np.exp(-np.linspace(-2,2,5)**2); g1 /= g1.sum()
K = np.outer(g1, g1)
full = conv2d_np(sample_img, K)
# separable: convolve rows with g1, then columns with g1
rows = np.array([np.convolve(r, g1, mode='valid') for r in sample_img])
sep  = np.array([np.convolve(rows[:,j], g1, mode='valid') for j in range(rows.shape[1])]).T
print('separable vs full max abs diff:', np.abs(full - sep).max())
ks = np.arange(3, 16, 2)
plt.figure(figsize=(7,4))
plt.plot(ks, ks**2, 'o-', label='naive  k²'); plt.plot(ks, 2*ks, 'o-', label='separable  2k')
plt.xlabel('kernel size k'); plt.ylabel('multiplies per pixel'); plt.title('Station 9 — separable convolution cost')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f'speed-up at k=9: {9**2/(2*9):.1f}x')

In [ ]:
# FFT convolution equals direct convolution (circular, with padding).
img = sample_img; ker = kernels['blur (box)']
Hh, Ww = img.shape; kh, kw = ker.shape
padded = np.zeros((Hh, Ww)); padded[:kh,:kw] = ker
fft_conv = np.real(np.fft.ifft2(np.fft.fft2(img) * np.fft.fft2(padded)))
# align direct (valid) conv into the same circular convention for comparison of the interior
direct = conv2d_np(img, ker)
print('FFT reproduces convolution (interior max diff):',
      f"{np.abs(fft_conv[kh-1:kh-1+direct.shape[0], kw-1:kw-1+direct.shape[1]] - direct).max():.2e}")
print('For large kernels the FFT turns O(n·k²) convolution into O(n log n).')

## Station 10 — Random / unsupervised features & normalization

**Random** conv filters already extract edges and textures — a classifier on random-conv + pooling features does respectably, useful for fast architecture search or scarce labels. Separately, **batch normalization** re-centres and re-scales each layer's activations (`x̂=(x−μ)/√(σ²+ε)`, then `γx̂+β`) so their distribution stays stable, speeding and steadying training.


In [ ]:
# Random filters still find structure, and make useful features for a linear classifier.
torch.manual_seed(0)
rand_conv = nn.Conv2d(1, 16, 5, padding=2); rand_conv.requires_grad_(False)   # never trained
with torch.no_grad(): rmaps = F.relu(rand_conv(torch.tensor(sample_img).view(1,1,28,28).float()))[0]
fig, ax = plt.subplots(1,8, figsize=(12,1.8))
for i in range(8):
    ax[i].imshow(rmaps[i], cmap='gray'); ax[i].axis('off')
plt.suptitle('Station 10 — feature maps from UNTRAINED random filters (edges appear)'); plt.show()

class RandomFeatureClf(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1,16,5,padding=2); self.conv.requires_grad_(False)  # frozen random
        self.fc = nn.Linear(16*14*14, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv(x)), 2); return self.fc(x.flatten(1))
torch.manual_seed(0); rf = RandomFeatureClf()
acc_rf, _ = train_clf(rf, epochs=5)
class PixelClf(nn.Module):
    def __init__(self): super().__init__(); self.fc = nn.Linear(784,10)
    def forward(self,x): return self.fc(x.flatten(1))
torch.manual_seed(0); acc_px, _ = train_clf(PixelClf(), epochs=5)
print(f'linear on raw pixels          : val acc {acc_px:.3f}')
print(f'linear on random-conv features: val acc {acc_rf:.3f}  <- random filters help')

In [ ]:
# Batch norm steadies and speeds training.
def cnn(bn):
    layers = [nn.Conv2d(1,16,3,padding=1)]
    if bn: layers.append(nn.BatchNorm2d(16))
    layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16,32,3,padding=1)]
    if bn: layers.append(nn.BatchNorm2d(32))
    layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(32*7*7,10)]
    return nn.Sequential(*layers)

plt.figure(figsize=(8,4.5))
for bn, c in [(False,'gray'), (True,'crimson')]:
    torch.manual_seed(0); _, losses = train_clf(cnn(bn), epochs=3, lr=3e-3, record=True)
    sm = np.convolve(losses, np.ones(20)/20, mode='valid')
    plt.plot(sm, color=c, label='with BatchNorm' if bn else 'no BatchNorm')
plt.xlabel('minibatch step'); plt.ylabel('training loss (smoothed)')
plt.title('Station 10 — BatchNorm trains faster and steadier'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## Station 11 — Computer vision & the ImageNet moment

Stacked conv–pool stages build a **feature hierarchy**: edges → textures → parts → objects, the receptive field growing with depth. ImageNet + GPUs let deep CNNs shine (AlexNet, 2012). We train a small CNN, compare it to an MLP with a similar budget, and visualize the learned first-layer filters and activations.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1); self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.fc = nn.Linear(32*7*7, 10)
    def forward(self, x, feats=False):
        a1 = F.relu(self.conv1(x)); p1 = F.max_pool2d(a1, 2)
        a2 = F.relu(self.conv2(p1)); p2 = F.max_pool2d(a2, 2)
        out = self.fc(p2.flatten(1))
        return (out, a1, a2) if feats else out

class MLP(nn.Module):
    def __init__(self): super().__init__(); self.net = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU(), nn.Linear(256,10))
    def forward(self,x): return self.net(x)

torch.manual_seed(0); cnn_model = SmallCNN(); acc_cnn,_ = train_clf(cnn_model, epochs=6)
torch.manual_seed(0); acc_mlp,_ = train_clf(MLP(), epochs=6)
np_ = lambda m: sum(p.numel() for p in m.parameters())
print(f'SmallCNN : val acc {acc_cnn:.3f} | params {np_(cnn_model):,}')
print(f'MLP      : val acc {acc_mlp:.3f} | params {np_(MLP()):,}')

In [ ]:
# Feature hierarchy: first-layer filters and the activations they produce.
W1 = cnn_model.conv1.weight.detach().cpu()
fig, ax = plt.subplots(2,8, figsize=(12,3.2))
for i in range(8):
    ax[0,i].imshow(W1[i,0], cmap='RdBu'); ax[0,i].axis('off')
ax[0,0].set_ylabel('conv1\nfilters')
xb,_ = next(iter(val_loader)); x = xb[:1].to(device)
with torch.no_grad(): _, a1, a2 = cnn_model(x, feats=True)
for i in range(8):
    ax[1,i].imshow(a1[0,i].cpu(), cmap='gray'); ax[1,i].axis('off')
ax[1,0].set_ylabel('conv1\nactivations')
plt.suptitle('Station 11 — learned first-layer filters (edge-like) and their feature maps'); plt.tight_layout(); plt.show()
print(f'conv1 receptive field 3x3; after pool+conv2 each unit sees a larger patch — features grow more abstract with depth.')

## Key takeaways

1. **Grid data → build in the structure.** Dense layers waste parameters and ignore geometry; convolution assumes local, translation-symmetric structure.
2. **Convolution = slide a kernel.** Sum input×kernel at each position to score a local pattern; sweep to get a feature map.
3. **Sparse · shared · equivariant.** Few connections per unit, one filter reused everywhere, outputs that shift with the input.
4. **Pooling → shift tolerance.** Max/avg over a patch summarizes it, so small shifts barely change the result and the map shrinks.
5. **Rooted in V1.** Oriented simple cells ≈ convolution; shift-tolerant complex cells ≈ pooling; learned filters ≈ Gabors.
6. **An infinitely strong prior.** Convolution is a dense layer forced to be local and weight-shared; pooling forces local translation invariance.
7. **Knobs & shapes.** Stride, padding, dilation, channels — with a clean output-size formula, verified against PyTorch.
8. **Structured outputs & channels.** Convolution preserves layout (segmentation) and always spans channels (4-D kernels).
9. **Fast & cheap.** Separable kernels (`k²`→`2k`) and FFTs speed convolution up.
10. **Random features & BatchNorm.** Untrained filters already find edges; BN keeps deep training stable.
11. **Feature hierarchy & ImageNet.** Edges → parts → objects; scale + GPUs made CNNs the default for vision.

### Try it yourself
- Add a third conv block to `SmallCNN` and watch accuracy and the receptive field grow.
- Replace max-pool with stride-2 convolutions and compare accuracy and parameter count.
- Turn the segmentation target into an edge map (`Sobel` magnitude thresholded) and retrain the FCN.

*Reference: Goodfellow, Bengio & Courville, "Deep Learning," Chapter 9.*
